# Score protein–ligand binder complexes with MaSIF-neosurface

Score candidate binders that are already aligned to a processed protein–ligand target.

This notebook previews the configuration by default. Edit your paths and set `RUN_TASK = True` when you are ready to run.


## 1. Before you start

- Install the `design` optional dependencies.
- Obtain `weights/neosurface/ipa_12A_surface_atom.pt`.
- Prepare a ligand-containing target packet and aligned candidate packets. Keep their referenced PDB files available.
- If your candidates are not aligned, use [seed docking](seed.ipynb) first, selecting `task: neosurface` and the neosurface checkpoint.

Use the `Python (masif_torch)` kernel. See [installation](../README.md#installation) and the [user guide](../USAGE.md).


## 2. Open the project


In [ ]:
from pathlib import Path
import importlib.util
import os
import shlex
import shutil
import subprocess
import sys
import tempfile

from omegaconf import OmegaConf

ROOT = next((path for path in (Path.cwd(), *Path.cwd().parents)
             if (path / "pyproject.toml").is_file() and (path / "examples").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook from the MaSIF_torch repository or its tutorials directory.")
os.chdir(ROOT)
CLI = [sys.executable, "-m", "masif"]
RUN_TASK = False  # Change to True only after editing the paths below.
print("Repository:", ROOT)
print("Python:", sys.executable)


## 3. Choose your inputs and settings

Paths below are relative to the repository unless you provide an absolute path. Choose a separate output directory for each run.


In [ ]:
TASK = "neosurface"
config = OmegaConf.create({
    "task": "neosurface", "action": "score",
    "target": "data/design/target_with_ligand.npz",
    "database": ["data/design/already_aligned_binder.npz"],
    "checkpoint": "weights/neosurface/ipa_12A_surface_atom.pt",
    "output": "results/tutorial_neosurface", "device": "cpu", "seed": 42, "num_sites": 1,
})
# Each source packet must already be in the target coordinate system.


## 4. Preview your configuration

This prints the Hydra configuration without loading your input data or running the task.


In [ ]:
config_directory = tempfile.TemporaryDirectory(prefix="masif_tutorial_")
config_path = Path(config_directory.name) / "task.yaml"
OmegaConf.save(config, config_path)
command = CLI + [TASK, "--config", str(config_path)]
print(shlex.join(command))
subprocess.run(command + ["--cfg", "job"], check=True)


## 5. Run when ready

After editing the settings, set `RUN_TASK = True` in the setup cell and rerun the notebook. The checks below help identify missing inputs before starting.


In [ ]:
if RUN_TASK:
    from masif_torch.shared.design.data import SurfaceRecord

    for filename in [config.target, config.checkpoint, *config.database]:
        if not Path(filename).is_file():
            raise FileNotFoundError(f"Missing {filename}. Prepare the target/aligned candidate or obtain the checkpoint first.")
    for packet in [config.target, *config.database]:
        surface = SurfaceRecord.from_npz(packet)
        if surface.pdb_path is None or not surface.pdb_path.is_file():
            raise FileNotFoundError(f"Packet {packet} needs an accessible referenced PDB. Repack it with the PDB available.")
    subprocess.run(command, check=True)
else:
    print("Preview only. Edit your paths and set RUN_TASK = True to run.")


## 6. Next steps

Use the chosen `output` folder for this run. See [Rosetta](rosetta.ipynb) to refine a positioned binder. Use [ligand preparation](prepare_ligand.ipynb) to prepare another protein–ligand target.
